# 01. Подготовка данных — Retail Customer Analytics

**Цель:** получить проверенную базу покупок для SQL-метрик, когорт и RFM.
Бизнес-задача — выбор аудитории для кампании повторных покупок.

Порядок работы: источник → качество → правила → строки покупок → заказы → сверки.
Основное окно: **[01.12.2010, 01.12.2011)**. Исходник не изменяем.

Запускайте ячейки по порядку из полного проекта. Архив содержит исходный Excel,
поэтому повторный запуск возможен без сети. При отсутствии исходника первая
ячейка загрузки получает официальный архив UCI. Суммы — GBP.

Источник: Chen, D. (2015). Online Retail [Dataset]. UCI Machine Learning Repository.
https://doi.org/10.24432/C5BW33, CC BY 4.0.
Документация: https://archive.ics.uci.edu/dataset/352/online+retail


In [1]:
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile
from urllib.request import urlopen
from datetime import datetime, timezone
from contextlib import closing
from tempfile import NamedTemporaryFile
import hashlib
import importlib.metadata
import json
import platform
import sqlite3

import numpy as np
import pandas as pd

# Работает из корня проекта и из notebooks/; абсолютных путей нет.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / 'docs/01_project_brief.md').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Откройте ноутбук внутри распакованного retail-customer-analytics.')

RAW = ROOT / 'data/raw/Online Retail.xlsx'
PROCESSED = ROOT / 'data/processed'
TABLES = ROOT / 'reports/tables'
for folder in [RAW.parent, PROCESSED, TABLES]:
    folder.mkdir(parents=True, exist_ok=True)

START = pd.Timestamp('2010-12-01')
CUTOFF = pd.Timestamp('2011-12-01')
SOURCE_URL = 'https://archive.ics.uci.edu/static/public/352/online%2Bretail.zip'
EXPECTED_SHA256 = '43465a06f2ccf7c8b5bd2892bc7defb52f97487934fe93b16ae4c3936424676d'
print('Python:', platform.python_version(), '| pandas:', pd.__version__)
print('Окно:', START.date(), '—', CUTOFF.date(), '(правая граница исключена)')


Python: 3.12.14 | pandas: 2.2.3
Окно: 2010-12-01 — 2011-12-01 (правая граница исключена)


## 1. Получение и проверка источника

Контрольная сумма привязывает результат к конкретным байтам Excel. Если файл
изменился, выполнение останавливается: новую версию нужно изучить и документировать.
Проверяем источник до записи скачанных байтов на диск.


In [2]:
if not RAW.exists():
    with urlopen(SOURCE_URL, timeout=60) as response:
        archive_bytes = response.read()
    with ZipFile(BytesIO(archive_bytes)) as archive:
        source_bytes = archive.read('Online Retail.xlsx')
    actual_hash = hashlib.sha256(source_bytes).hexdigest()
    if actual_hash != EXPECTED_SHA256:
        raise ValueError('Загруженный источник отличается от проверенной версии.')
    RAW.write_bytes(source_bytes)

source_hash = hashlib.sha256(RAW.read_bytes()).hexdigest()
assert source_hash == EXPECTED_SHA256, 'Excel отличается от проверенного источника.'
excel = pd.ExcelFile(RAW, engine='openpyxl')
assert excel.sheet_names == ['Online Retail'], excel.sheet_names
raw = pd.read_excel(excel, sheet_name='Online Retail')
excel.close()
expected_columns = ['InvoiceNo', 'StockCode', 'Description', 'Quantity',
                    'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']
assert raw.columns.tolist() == expected_columns
assert len(raw) == 541909
assert raw['CustomerID'].dropna().mod(1).eq(0).all()

print('SHA-256:', source_hash)
print('Строк:', len(raw), '| полей:', raw.shape[1])
print('Даты:', raw['InvoiceDate'].min(), '—', raw['InvoiceDate'].max())
print(raw.head(3).to_string(index=False))


SHA-256: 43465a06f2ccf7c8b5bd2892bc7defb52f97487934fe93b16ae4c3936424676d
Строк: 541909 | полей: 8
Даты: 2010-12-01 08:26:00 — 2011-12-09 12:50:00
InvoiceNo StockCode                        Description  Quantity         InvoiceDate  UnitPrice  CustomerID        Country
   536365    85123A WHITE HANGING HEART T-LIGHT HOLDER         6 2010-12-01 08:26:00       2.55     17850.0 United Kingdom
   536365     71053                WHITE METAL LANTERN         6 2010-12-01 08:26:00       3.39     17850.0 United Kingdom
   536365    84406B     CREAM CUPID HEARTS COAT HANGER         8 2010-12-01 08:26:00       2.75     17850.0 United Kingdom


## 2. Профиль качества исходного файла

Одна строка — одна позиция счёта. Пропуск клиента не означает отсутствие продажи.
Проверки ниже относятся ко всему исходнику, включая неполный декабрь 2011 года.
Флаги могут пересекаться: например, запись отмены имеет отрицательное количество.
Их количества нельзя складывать как взаимоисключающие причины удаления.


In [3]:
missing = pd.DataFrame({
    'field': raw.columns,
    'dtype': raw.dtypes.astype(str).to_numpy(),
    'missing_rows': raw.isna().sum().to_numpy(),
    'missing_pct': (100 * raw.isna().mean()).to_numpy(),
})
raw_duplicate = raw.duplicated(subset=expected_columns, keep='first')
raw_cancel = raw['InvoiceNo'].astype('string').str.upper().str.startswith('C', na=False)
raw_checks = pd.DataFrame([
    ('exact_duplicate_beyond_first', int(raw_duplicate.sum())),
    ('cancellation_prefix', int(raw_cancel.sum())),
    ('negative_quantity', int(raw['Quantity'].lt(0).sum())),
    ('zero_quantity', int(raw['Quantity'].eq(0).sum())),
    ('negative_unit_price', int(raw['UnitPrice'].lt(0).sum())),
    ('zero_unit_price', int(raw['UnitPrice'].eq(0).sum())),
], columns=['check', 'rows'])
missing.to_csv(TABLES / 'raw_missing_values.csv', index=False)
raw_checks.to_csv(TABLES / 'raw_quality_checks.csv', index=False)
print(missing.to_string(index=False, float_format=lambda x: f'{x:.3f}'))
print('\nПересекающиеся признаки:')
print(raw_checks.to_string(index=False))


      field          dtype  missing_rows  missing_pct
  InvoiceNo         object             0        0.000
  StockCode         object             0        0.000
Description         object          1454        0.268
   Quantity          int64             0        0.000
InvoiceDate datetime64[ns]             0        0.000
  UnitPrice        float64             0        0.000
 CustomerID        float64        135080       24.927
    Country         object             0        0.000

Пересекающиеся признаки:
                       check  rows
exact_duplicate_beyond_first  5268
         cancellation_prefix  9288
           negative_quantity 10624
               zero_quantity     0
         negative_unit_price     2
             zero_unit_price  2515


## 3. Приведение типов и признаки аудита

Номер исходной строки Excel (`source_row`, первая строка данных = 2) позволяет
найти любое включённое или исключённое наблюдение. Сырой Excel остаётся неизменным.
Идентификаторы переводим в строки, пропуски сохраняем. `CustomerID` не числовой признак.

Цены сохраняем с исходной точностью, включая 0.001 GBP. Округление до пенсов
используем только при выводе итогов. Денежные сверки допускают погрешность 0.01 GBP.


In [4]:
def as_id(series):
    # CustomerID читается из Excel как float из-за пропусков: 17850.0 → "17850".
    return (series.astype('string').str.strip()
            .str.replace(r'^(\d+)\.0$', r'\1', regex=True)
            .replace('', pd.NA))

df = raw.rename(columns={
    'InvoiceNo': 'invoice_no', 'StockCode': 'stock_code',
    'Description': 'description', 'Quantity': 'quantity',
    'InvoiceDate': 'invoice_date', 'UnitPrice': 'unit_price',
    'CustomerID': 'customer_id', 'Country': 'country',
}).copy()
df.insert(0, 'source_row', np.arange(2, len(df) + 2))
for col in ['invoice_no', 'stock_code', 'customer_id']:
    df[col] = as_id(df[col])
df['stock_code'] = df['stock_code'].str.upper()
df['country'] = df['country'].astype('string').str.strip().replace('', pd.NA)
df['invoice_date'] = pd.to_datetime(df['invoice_date'], errors='coerce')
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
df['unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce')
df['line_amount_gbp'] = df['quantity'] * df['unit_price']
df['is_exact_duplicate'] = raw_duplicate.astype(int)
df['is_cancellation'] = df['invoice_no'].str.upper().str.startswith('C', na=False)
df['in_analysis_window'] = df['invoice_date'].ge(START) & df['invoice_date'].lt(CUTOFF)
df['valid_required'] = (
    df[['invoice_no', 'stock_code', 'invoice_date', 'quantity', 'unit_price']].notna().all(axis=1)
    & np.isfinite(df['quantity']) & np.isfinite(df['unit_price'])
)
print('Некорректные обязательные поля:', int((~df['valid_required']).sum()))
print('Строк в основном окне:', int(df['in_analysis_window'].sum()))


Некорректные обязательные поля: 0
Строк в основном окне: 516384


## 4. Товарные и служебные позиции

Ниже — явный список исключаемых кодов, составленный по описаниям в исходнике.
Это наша аналитическая классификация, а не официальный справочник категорий UCI.
Доставка, комиссии, скидки, Manual, Adjust bad debt и Samples не считаются
обычными товарными продажами. Подарочные сертификаты выделяем отдельно, чтобы
не смешивать оплату сертификата с покупкой физического товара.

Формат кода сам по себе не является правилом удаления: DCGS-коды с названиями
товаров и PADS оставляем. Пропуск Description не удаляет строку автоматически.


In [5]:
excluded_codes = {
    'POST': 'delivery', 'DOT': 'delivery', 'C2': 'delivery',
    'BANK CHARGES': 'fee', 'AMAZONFEE': 'fee', 'CRUK': 'commission',
    'D': 'discount', 'M': 'manual', 'B': 'bad_debt_adjustment', 'S': 'samples',
}
df['excluded_code_type'] = df['stock_code'].map(excluded_codes).astype('string')
gift = df['stock_code'].str.startswith('GIFT_', na=False)
df.loc[gift, 'excluded_code_type'] = 'gift_voucher'
df['is_excluded_code'] = df['excluded_code_type'].notna()

special = df.loc[~df['stock_code'].str.fullmatch(r'\d{5}[A-Z]{0,2}', na=False)].copy()
special_codes = special.groupby('stock_code', dropna=False).agg(
    rows=('source_row', 'size'), description_example=('description', 'first'),
    excluded_code_type=('excluded_code_type', 'first'),
    signed_amount_all_dates_gbp=('line_amount_gbp', 'sum'),
).reset_index()
special_codes.to_csv(TABLES / 'special_stock_codes.csv', index=False)
print(special_codes[['stock_code', 'rows', 'description_example', 'excluded_code_type']].to_string(index=False))


  stock_code  rows                description_example  excluded_code_type
   AMAZONFEE    34                         AMAZON FEE                 fee
           B     3                    Adjust bad debt bad_debt_adjustment
BANK CHARGES    37                       Bank Charges                 fee
          C2   144                           CARRIAGE            delivery
        CRUK    16                    CRUK Commission          commission
           D    77                           Discount            discount
    DCGS0003     5                BOXED GLASS ASHTRAY                <NA>
    DCGS0004     1         HAYNES CAMPER SHOULDER BAG                <NA>
    DCGS0055     1                               None                <NA>
    DCGS0057     1                               None                <NA>
   DCGS0066P     1                               None                <NA>
    DCGS0067     1                               ebay                <NA>
    DCGS0068     1                    

## 5. Согласованность счетов и выбор строк покупок

Проверяем счёт до отбора товарных позиций, чтобы служебная строка не скрыла
противоречие клиента или страны. Разные минуты внутри одного дня допустимы,
если клиент и страна согласованы. Время заказа — самое раннее время его
включённой товарной позиции. Если бы были разные дни/клиенты/страны внутри
счёта, его товарные строки попали бы в исключения с отдельной причиной.

В каждой строке аудита есть одна основная причина исключения. Причины присваиваются
по приоритету ниже, поэтому декабрьская отмена получает `outside_window`.
Полный флаг `is_cancellation` сохраняется независимо от этого приоритета.


In [6]:
invoice_profile = df.groupby('invoice_no', dropna=False).agg(
    customer_values=('customer_id', lambda s: s.nunique(dropna=False)),
    country_values=('country', lambda s: s.nunique(dropna=False)),
    min_date=('invoice_date', 'min'), max_date=('invoice_date', 'max'),
    timestamp_values=('invoice_date', 'nunique'),
)
invoice_profile['same_day'] = (
    invoice_profile['min_date'].dt.normalize() == invoice_profile['max_date'].dt.normalize()
)
invoice_profile['is_consistent'] = (
    invoice_profile['customer_values'].eq(1)
    & invoice_profile['country_values'].eq(1) & invoice_profile['same_day']
)
df['consistent_invoice'] = df['invoice_no'].map(invoice_profile['is_consistent']).fillna(False).astype(bool)

conditions = [
    ~df['valid_required'], ~df['in_analysis_window'], df['is_cancellation'],
    df['quantity'].le(0), df['unit_price'].le(0), df['is_excluded_code'],
    ~df['consistent_invoice'],
]
reasons = ['invalid_required_fields', 'outside_window', 'cancellation',
           'non_positive_quantity', 'non_positive_price', 'excluded_code', 'inconsistent_invoice']
df['status'] = np.select([s.to_numpy(dtype=bool) for s in conditions], reasons, default='included')
purchase_lines = df.loc[df['status'].eq('included')].copy()
assert purchase_lines['quantity'].gt(0).all()
assert purchase_lines['unit_price'].gt(0).all()
assert not purchase_lines['is_cancellation'].any()
assert purchase_lines['in_analysis_window'].all()

decision_counts = df.groupby('status', sort=False).agg(
    rows=('source_row', 'size'), signed_amount_gbp=('line_amount_gbp', 'sum')
).reset_index()
decision_counts['rows_pct_of_raw'] = 100 * decision_counts['rows'] / len(raw)
assert decision_counts['rows'].sum() == len(raw)
decision_counts.to_csv(TABLES / 'cleaning_decisions.csv', index=False)
invoice_profile.reset_index().to_csv(TABLES / 'invoice_consistency.csv', index=False)
print(decision_counts.to_string(index=False, float_format=lambda x: f'{x:.3f}'))
print('Включено товарных позиций:', len(purchase_lines))
print('Несогласованных счетов в исходнике:', int((~invoice_profile['is_consistent']).sum()))


               status   rows  signed_amount_gbp  rows_pct_of_raw
             included 502730        9655939.793           92.770
        excluded_code   2263         371952.071            0.418
         cancellation   8928        -691687.820            1.648
   non_positive_price   1157         -22124.120            0.214
non_positive_quantity   1306              0.000            0.241
       outside_window  25525         433668.010            4.710
Включено товарных позиций: 502730
Несогласованных счетов в исходнике: 0


## 6. Повторы: основной вариант и чувствительность

В основной версии **сохраняем точные повторы**: нет независимого номера позиции,
поэтому одинаковые строки могут отражать отдельные записи одного счёта.
Это консервативное допущение, а не доказательство правильности каждого повтора.

Альтернативный вариант удаляет строки, полностью совпадающие по всем восьми
исходным полям, кроме первого вхождения. Не удаляем повтор лишь по паре
InvoiceNo–StockCode. Показываем изменение суммы и количества позиций; на следующих
этапах проверим устойчивость ключевых клиентских выводов к этому решению.


In [7]:
without_duplicates = purchase_lines.loc[purchase_lines['is_exact_duplicate'].eq(0)]
duplicate_sensitivity = pd.DataFrame([
    {'variant': 'keep_exact_duplicates', 'lines': len(purchase_lines),
     'orders': purchase_lines['invoice_no'].nunique(),
     'gross_sales_gbp': purchase_lines['line_amount_gbp'].sum()},
    {'variant': 'drop_exact_duplicates', 'lines': len(without_duplicates),
     'orders': without_duplicates['invoice_no'].nunique(),
     'gross_sales_gbp': without_duplicates['line_amount_gbp'].sum()},
])
duplicate_delta = float(purchase_lines['line_amount_gbp'].sum() - without_duplicates['line_amount_gbp'].sum())
duplicate_delta_pct = 100 * duplicate_delta / purchase_lines['line_amount_gbp'].sum()
duplicate_sensitivity.to_csv(TABLES / 'duplicate_sensitivity.csv', index=False)
print(duplicate_sensitivity.to_string(index=False, float_format=lambda x: f'{x:.3f}'))
print(f'Разница суммы: {duplicate_delta:,.2f} GBP ({duplicate_delta_pct:.3f}%)')


              variant  lines  orders  gross_sales_gbp
keep_exact_duplicates 502730   18957      9655939.793
drop_exact_duplicates 497787   18957      9632728.403
Разница суммы: 23,211.39 GBP (0.240%)


## 7. Экспорт в SQLite и сборка заказов через SQL

База хранит исходные поля, аудит всех строк и строки товарных покупок.
`excluded_lines` и `cancellation_lines` — представления аудита, а не дополнительные
копии данных. `orders` создаётся запросом из `sql/01_orders.sql`.
`customer_orders` оставляет только известный CustomerID.

Создаём новую базу во временном файле. Только после успешных проверок она заменит
предыдущую версию `retail.sqlite`. Сырой файл и отчёты не являются источником
бизнес-агрегаций: они служат для трассировки и проверки.


In [8]:
DB = PROCESSED / 'retail.sqlite'
# Отдельный временный файл для каждого запуска исключает столкновение сборок.
with NamedTemporaryFile(prefix='retail.build.', suffix='.sqlite', dir=PROCESSED, delete=False) as temp:
    TEMP_DB = Path(temp.name)

raw_for_sql = raw.copy()
raw_for_sql.insert(0, 'source_row', df['source_row'])
for col in ['InvoiceNo', 'StockCode', 'CustomerID']:
    raw_for_sql[col] = as_id(raw_for_sql[col])

line_columns = ['source_row', 'invoice_no', 'stock_code', 'description', 'quantity',
                'invoice_date', 'unit_price', 'customer_id', 'country',
                'line_amount_gbp', 'is_exact_duplicate']
# Сборка в памяти; на диск записывается уже завершённая SQLite-база.
with closing(sqlite3.connect(':memory:')) as conn:
    raw_for_sql.to_sql('raw_transactions', conn, if_exists='replace', index=False, chunksize=10000)
    df.to_sql('transaction_audit', conn, if_exists='replace', index=False, chunksize=10000)
    purchase_lines[line_columns].to_sql('purchase_lines', conn, if_exists='replace', index=False, chunksize=10000)
    conn.executescript("""
        DROP VIEW IF EXISTS excluded_lines;
        DROP VIEW IF EXISTS cancellation_lines;
        CREATE UNIQUE INDEX idx_raw_source_row ON raw_transactions(source_row);
        CREATE UNIQUE INDEX idx_audit_source_row ON transaction_audit(source_row);
        CREATE UNIQUE INDEX idx_lines_source_row ON purchase_lines(source_row);
        CREATE INDEX idx_lines_invoice ON purchase_lines(invoice_no);
        CREATE INDEX idx_audit_status ON transaction_audit(status);
        CREATE VIEW excluded_lines AS SELECT * FROM transaction_audit WHERE status != 'included';
        CREATE VIEW cancellation_lines AS SELECT * FROM transaction_audit WHERE is_cancellation = 1;
    """)
    conn.executescript((ROOT / 'sql/01_orders.sql').read_text(encoding='utf-8'))
    orders = pd.read_sql_query('SELECT * FROM orders ORDER BY invoice_no', conn)
    customer_orders = pd.read_sql_query('SELECT * FROM customer_orders ORDER BY invoice_no', conn)
    schema_objects = pd.read_sql_query(
        "SELECT type, name FROM sqlite_master WHERE type IN ('table','view') ORDER BY type, name", conn
    )
    conn.commit()
    TEMP_DB.write_bytes(conn.serialize())
print(schema_objects.to_string(index=False))
print('Заказов:', len(orders), '| с известным клиентом:', len(customer_orders))


 type               name
table             orders
table     purchase_lines
table   raw_transactions
table  transaction_audit
 view cancellation_lines
 view    customer_orders
 view     excluded_lines
Заказов: 18957 | с известным клиентом: 17626


## 8. Независимые сверки SQL и pandas

Пересчитываем заказы независимо через pandas. Сверяем не только общую сумму,
но каждый счёт и каждый месяц — ошибка в одном заказе не должна скрываться
за противоположной ошибкой в другом. Проверяем также уникальность, сохранность
всех исходных строк и отсутствие служебных кодов в покупках.


In [9]:
pandas_orders = purchase_lines.groupby('invoice_no').agg(
    gross_sales_gbp=('line_amount_gbp', 'sum'), units=('quantity', 'sum'),
    line_count=('source_row', 'size'), order_date=('invoice_date', 'min'),
).sort_index()
sql_orders = orders.set_index('invoice_no').sort_index()
assert pandas_orders.index.tolist() == sql_orders.index.tolist()
assert np.allclose(pandas_orders['gross_sales_gbp'], sql_orders['gross_sales_gbp'], rtol=0, atol=0.01)
for col in ['units', 'line_count']:
    assert np.array_equal(pandas_orders[col], sql_orders[col])
assert np.array_equal(pandas_orders['order_date'].to_numpy(), pd.to_datetime(sql_orders['order_date']).to_numpy())
assert orders['invoice_no'].is_unique
assert purchase_lines['source_row'].is_unique
assert int(orders['line_count'].sum()) == len(purchase_lines)
assert np.isclose(orders['gross_sales_gbp'].sum(), purchase_lines['line_amount_gbp'].sum(), rtol=0, atol=0.01)

with closing(sqlite3.connect(TEMP_DB)) as conn:
    assert conn.execute('PRAGMA integrity_check').fetchone()[0] == 'ok'
    assert conn.execute('SELECT COUNT(*) FROM raw_transactions').fetchone()[0] == len(raw)
    assert conn.execute('SELECT COUNT(*) FROM transaction_audit').fetchone()[0] == len(raw)
    assert conn.execute('SELECT COUNT(*) FROM excluded_lines').fetchone()[0] + len(purchase_lines) == len(raw)
    assert conn.execute("SELECT COUNT(*) FROM purchase_lines WHERE quantity <= 0 OR unit_price <= 0 OR UPPER(invoice_no) LIKE 'C%'").fetchone()[0] == 0
    sql_months = pd.read_sql_query("""
        SELECT substr(order_date, 1, 7) AS month, COUNT(*) AS orders,
               SUM(gross_sales_gbp) AS gross_sales_gbp, SUM(units) AS units
        FROM orders GROUP BY month ORDER BY month
    """, conn).set_index('month')

month_labels = pd.to_datetime(orders['order_date']).dt.strftime('%Y-%m')
pandas_months = orders.assign(month=month_labels).groupby('month').agg(
    orders=('invoice_no', 'size'), gross_sales_gbp=('gross_sales_gbp', 'sum'), units=('units', 'sum'))
assert len(sql_months) == 12
assert sql_months.index.tolist() == pd.period_range(START, CUTOFF - pd.Timedelta(days=1), freq='M').astype(str).tolist()
assert sql_months.index.equals(pandas_months.index)
assert np.array_equal(sql_months['orders'], pandas_months['orders'])
assert np.array_equal(sql_months['units'], pandas_months['units'])
assert np.allclose(sql_months['gross_sales_gbp'], pandas_months['gross_sales_gbp'], rtol=0, atol=0.01)
sql_months.to_csv(TABLES / 'monthly_reconciliation.csv')

# Все соединения закрыты, проверки прошли. Замена выполняется в той же папке.
TEMP_DB.replace(DB)
print('Проверки пройдены: каждый заказ, 12 месяцев, полнота строк, уникальность и SQLite integrity_check.')


Проверки пройдены: каждый заказ, 12 месяцев, полнота строк, уникальность и SQLite integrity_check.


## 9. Покрытие клиентского анализа и записи отмен

Клиентский анализ относится к идентифицированной части базы. Показатель покрытия
считаем отдельно по заказам и по их сумме. Он не равен доле непустых строк.
Таблица отмен включает только основное окно; её signed amount — сумма записей
отмен, а не доказанная сумма возвратов первоначальных заказов.

Очень крупные заказы не удаляем автоматически. Сохраняем топ-10 для проверки
влияния на будущие выводы. При интерпретации Monetary учитываем отмены отдельно.


In [10]:
gross_sales = float(orders['gross_sales_gbp'].sum())
identified_sales = float(customer_orders['gross_sales_gbp'].sum())
order_coverage_pct = 100 * len(customer_orders) / len(orders)
sales_coverage_pct = 100 * identified_sales / gross_sales
coverage = pd.DataFrame([
    ('orders', len(orders), len(customer_orders), order_coverage_pct),
    ('gross_sales_gbp', gross_sales, identified_sales, sales_coverage_pct),
], columns=['metric', 'all', 'identified', 'identified_pct'])
coverage.to_csv(TABLES / 'customer_id_coverage.csv', index=False)

cancellations = df.loc[df['is_cancellation'] & df['in_analysis_window']].copy()
cancel_goods = cancellations.loc[~cancellations['is_excluded_code']]
top_orders = orders.nlargest(10, 'gross_sales_gbp')
top_orders.to_csv(TABLES / 'largest_orders.csv', index=False)

# Диагностика клиента крупнейшего заказа: это не алгоритм связывания возвратов.
largest_invoice = top_orders.iloc[0]['invoice_no']
largest_customer = top_orders.iloc[0]['customer_id']
largest_date = pd.Timestamp(top_orders.iloc[0]['order_date'])
nearby_customer_records = df.loc[
    df['customer_id'].eq(largest_customer)
    & df['invoice_date'].dt.normalize().eq(largest_date.normalize()),
    ['source_row', 'invoice_no', 'stock_code', 'quantity', 'unit_price', 'line_amount_gbp', 'invoice_date']
]
nearby_customer_records.to_csv(TABLES / 'largest_order_same_day_records.csv', index=False)
print(coverage.to_string(index=False, float_format=lambda x: f'{x:.3f}'))
print('\nЗаписи отмен в окне:', len(cancellations), '| счетов отмен:', cancellations['invoice_no'].nunique())
print('Signed amount товарных записей отмен:', round(cancel_goods['line_amount_gbp'].sum(), 3), 'GBP')
print('\nПервые 5 крупнейших заказов:')
print(top_orders[['invoice_no', 'customer_id', 'order_date', 'gross_sales_gbp', 'units']].head(5).to_string(index=False))
print('\nЗаписи клиента крупнейшего заказа в тот же день:')
print(nearby_customer_records.to_string(index=False))


         metric         all  identified  identified_pct
         orders   18957.000   17626.000          92.979
gross_sales_gbp 9655939.793 8247836.223          85.417

Записи отмен в окне: 8928 | счетов отмен: 3690
Signed amount товарных записей отмен: -304597.42 GBP

Первые 5 крупнейших заказов:
invoice_no customer_id          order_date  gross_sales_gbp  units
    541431       12346 2011-01-18 10:01:00         77183.60  74215
    574941        None 2011-11-07 17:42:00         52940.94  14149
    576365        None 2011-11-14 17:55:00         50653.91  13956
    556444       15098 2011-06-10 15:28:00         38970.00     60
    567423       17450 2011-09-20 11:05:00         31698.16  12572

Записи клиента крупнейшего заказа в тот же день:
 source_row invoice_no stock_code  quantity  unit_price  line_amount_gbp        invoice_date
      61621     541431      23166     74215        1.04          77183.6 2011-01-18 10:01:00
      61626    C541433      23166    -74215        1.04        

## 10. Фиксация источника, результатов и правил

Компактные CSV-таблицы выше — результаты диагностики. Все строки, включая
исключённые, доступны в SQLite. Ниже создаются машинный JSON и читаемый отчёт.
Время проверки источника не меняем при повторном запуске того же файла.


In [11]:
manifest_path = RAW.parent / 'source_manifest.json'
previous_manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
verified_at = previous_manifest.get('verified_at_utc', datetime.now(timezone.utc).isoformat())
manifest = {
    'dataset': 'UCI Online Retail', 'dataset_id': 352,
    'dataset_doi': '10.24432/C5BW33', 'license': 'CC BY 4.0',
    'source_url': SOURCE_URL, 'file': RAW.name, 'sha256': source_hash,
    'size_bytes': RAW.stat().st_size, 'verified_at_utc': verified_at,
    'sheet': 'Online Retail', 'rows': len(raw),
    'min_timestamp': str(raw['InvoiceDate'].min()), 'max_timestamp': str(raw['InvoiceDate'].max()),
}
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
summary = {
    'analysis_start': str(START.date()), 'analysis_end_exclusive': str(CUTOFF.date()),
    'raw_rows': len(raw), 'raw_missing_customer_id': int(raw['CustomerID'].isna().sum()),
    'raw_missing_customer_id_pct': float(raw['CustomerID'].isna().mean() * 100),
    'raw_missing_description': int(raw['Description'].isna().sum()),
    'raw_duplicate_beyond_first': int(raw_duplicate.sum()),
    'raw_cancellation_rows': int(raw_cancel.sum()),
    'purchase_lines': len(purchase_lines), 'orders': len(orders),
    'identified_orders': len(customer_orders), 'identified_customers': int(customer_orders['customer_id'].nunique()),
    'gross_sales_gbp': gross_sales, 'identified_gross_sales_gbp': identified_sales,
    'order_coverage_pct': order_coverage_pct, 'sales_coverage_pct': sales_coverage_pct,
    'included_duplicate_rows': int(purchase_lines['is_exact_duplicate'].sum()),
    'duplicate_sales_delta_gbp': duplicate_delta, 'duplicate_sales_delta_pct': duplicate_delta_pct,
    'retained_orders_multiple_timestamps': int(orders['timestamp_count'].gt(1).sum()),
    'cancellation_rows_in_window': len(cancellations),
    'cancellation_invoices_in_window': int(cancellations['invoice_no'].nunique()),
    'goods_cancellation_signed_amount_gbp': float(cancel_goods['line_amount_gbp'].sum()),
    'largest_order_gbp': float(top_orders.iloc[0]['gross_sales_gbp']),
    'largest_invoice_no': largest_invoice,
    'source_sha256': source_hash,
    'checks_passed': ['row_accounting', 'invoice_and_month_reconciliation', 'order_uniqueness', 'sqlite_integrity'],
    'cleaning_counts': {r['status']: int(r['rows']) for r in decision_counts.to_dict('records')},
    'versions': {name: importlib.metadata.version(name) for name in ['pandas', 'numpy', 'openpyxl']},
}
(ROOT / 'reports/data_quality.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

def fmt(value, decimals=0):
    return f'{value:,.{decimals}f}'.replace(',', ' ')

reason_labels = {
    'included': 'Включённые товарные позиции', 'outside_window': 'Вне основного окна',
    'cancellation': 'Отмены внутри окна', 'non_positive_quantity': 'Остальные неположительные количества',
    'non_positive_price': 'Неположительные цены', 'excluded_code': 'Служебные коды и сертификаты',
    'invalid_required_fields': 'Некорректные обязательные поля', 'inconsistent_invoice': 'Несогласованные счета',
}
reason_table = '\n'.join(
    f"| {reason_labels[r['status']]} | {fmt(r['rows'])} | {fmt(r['rows_pct_of_raw'], 3)}% |"
    for r in decision_counts.to_dict('records')
)
report = f"""# Retail Customer Analytics — этап 2: качество и подготовка данных

Проверен официальный Excel, выполнены все кодовые ячейки ноутбука подготовки.
Основное окно: **01.12.2010–30.11.2011**. Исходный файл не изменён.

## Исходник

| Проверка | Результат |
| --- | ---: |
| Строки | {fmt(len(raw))} |
| Пропуски CustomerID | {fmt(summary['raw_missing_customer_id'])} ({fmt(summary['raw_missing_customer_id_pct'], 2)}%) |
| Пропуски Description | {fmt(summary['raw_missing_description'])} |
| Точные повторы сверх первого | {fmt(summary['raw_duplicate_beyond_first'])} |
| Строки с префиксом отмены C | {fmt(summary['raw_cancellation_rows'])} |

Даты исходника: {manifest['min_timestamp']}–{manifest['max_timestamp']}.
На карточке источника заявлено отсутствие пропусков, однако проверка файла обнаружила пропуски CustomerID и Description.

## Отбор строк

Причины ниже взаимоисключающие и присваиваются по приоритету: обязательные поля → окно → отмена → количество → цена → код → согласованность счёта.

| Решение | Строк | Доля исходника |
| --- | ---: | ---: |
{reason_table}

Независимые флаги сохраняются в аудите. Например, отмена за декабрь 2011 входит в «Вне основного окна» и одновременно имеет is_cancellation=1.

## Подготовленная база

| Показатель | Значение |
| --- | ---: |
| Товарные позиции | {fmt(len(purchase_lines))} |
| Заказы | {fmt(len(orders))} |
| Заказы с известным клиентом | {fmt(len(customer_orders))} |
| Идентифицированные клиенты | {fmt(summary['identified_customers'])} |
| Положительные товарные продажи, GBP | {fmt(gross_sales, 2)} |
| Покрытие заказов известным клиентом | {fmt(order_coverage_pct, 2)}% |
| Покрытие суммы продаж известным клиентом | {fmt(sales_coverage_pct, 2)}% |

Продажи включают строки без CustomerID. Когорты и RFM будут считаться только по идентифицированной части базы. Monetary — сумма положительных товарных продаж до вычета отмен за окно, а не чистая выручка или LTV.

## Значимые решения

- Точные повторы сохранены. Удаление повторов из включённых покупок уменьшает число позиций на {fmt(summary['included_duplicate_rows'])}, сумму на {fmt(duplicate_delta, 2)} GBP ({fmt(duplicate_delta_pct, 3)}%). Число заказов не меняется. В будущих выводах проверим чувствительность клиентских метрик.
- Коды POST, DOT, C2, BANK CHARGES, AMAZONFEE, CRUK, D, M, B, S и GIFT_* исключены из товарных продаж. Это классификация проекта по описаниям позиций. Необычные DCGS-коды с названиями товаров и PADS оставлены.
- {fmt(summary['retained_orders_multiple_timestamps'])} включённых счетов имеют разные времена позиций в пределах одного дня. Наблюдаемое расхождение — одна минута. Счёт остаётся одним заказом; время — минимум по товарным строкам.
- Крупные заказы сохранены. Крупнейший товарный счёт {largest_invoice} на {fmt(summary['largest_order_gbp'], 2)} GBP сопровождается записью отмены той же позиции у того же клиента в тот же день. CSV largest_order_same_day_records показывает исходные записи. Это существенное ограничение gross Monetary; приоритет кампании нельзя обосновывать только этой суммой.
- В основном окне {fmt(len(cancellations))} строк / {fmt(summary['cancellation_invoices_in_window'])} счетов отмен. Signed amount товарных записей отмен: {fmt(summary['goods_cancellation_signed_amount_gbp'], 2)} GBP. Сумма не называется объёмом возвратов конкретных исходных заказов: автоматического сопоставления нет.
- Цены 0.001 GBP сохранены. Сверки денежных сумм используют абсолютную погрешность 0.01 GBP.

## Что проверено

Сохранность всех {fmt(len(raw))} строк в raw_transactions и transaction_audit; разбиение на включённые/исключённые; допустимость покупок; уникальность исходных строк и заказов; совпадение каждого заказа и каждого из 12 месяцев в SQL/pandas; SQLite integrity_check. Проверки прошли.

## Файлы и воспроизведение

- notebooks/01_data_preparation.ipynb — объяснения, очистка, SQL-экспорт и сверки.
- sql/01_orders.sql — сборка заказов.
- data/processed/retail.sqlite — исходные поля, аудит, покупки, заказы и представления.
- reports/tables/ — профиль пропусков, причины отбора, справочник специальных кодов, чувствительность повторов, покрытие, крупные заказы и месячные сверки.
- data/raw/source_manifest.json — источник и SHA-256.

Архив проекта содержит Excel и SQLite для локальной работы. В Git исходник и восстановимая база исключены .gitignore. Для пересборки выполните ноутбук по порядку; интернет не нужен, пока Excel присутствует.

SHA-256 исходного Excel: `{source_hash}`.

Источник: Chen, D. (2015). Online Retail. UCI Machine Learning Repository. https://doi.org/10.24432/C5BW33. CC BY 4.0.

Следующий этап: содержательный анализ динамики и клиентских метрик. Числа здесь являются контролями подготовленной базы, а не готовыми маркетинговыми рекомендациями.
"""
(ROOT / 'reports/data_quality.md').write_text(report, encoding='utf-8')
print('Сохранены retail.sqlite, source_manifest.json, отчёт качества и диагностические CSV.')
print('Следующий этап: SQL-метрики и клиентская аналитика.')


Сохранены retail.sqlite, source_manifest.json, отчёт качества и диагностические CSV.
Следующий этап: SQL-метрики и клиентская аналитика.


## Вывод этапа подготовки

Теперь база расчётов определена и проверена. Суммы без ID входят в показатели
магазина, но не в клиентскую аналитику. Отмены и служебные позиции доступны для
аудита. Повторы сохранены с измеренной чувствительностью.

Откройте `reports/data_quality.md`, чтобы увидеть фактические числа и ограничения.
На следующем этапе используем эту базу для содержательного анализа, а не повторяем
очистку в другом ноутбуке.
